### By running the fgollowing cell with different values of delta 
### you can see that delta influences the number of solutions chosen as x equal to the null vector

In [3]:
import torch
import cvxpy as cp
import numpy as np
import pandas as pd
from IPython.display import display

def knapsack_specialized_pruning_sparse_leonardo(xi, v, w, C, device, delta):
    xi = xi.to(dtype=torch.float32, device=device)
    v = v.to(dtype=torch.float32, device=device)
    w = w.to(dtype=torch.float32, device=device)

    xi = xi - delta
    M = w.shape[0]

    b_list = []
    b = 0
    while True:
        delta_xi = xi[b + 1:] - xi[b]
        delta_v = v[b + 1:] - v[b]
        b = torch.argmin(delta_xi / delta_v) + 1 + b_list[-1] if b_list else 0
        if b != C - 1:
            b_list.append(int(b))
        if b + 1 > C - 1:
            break
    b_list.append(C - 1)

    x_plus = torch.zeros(C, dtype=torch.int32, device=device)
    x_plus[torch.tensor(b_list, device=device, dtype=torch.long)] = 1

    ratio = xi / v
    neg_indices = torch.where(v < 0)[0]
    pos_indices = torch.where(v >= 0)[0]
    neg_sorted = neg_indices[torch.argsort(ratio[neg_indices], descending=True)]
    pos_sorted = pos_indices[torch.argsort(ratio[pos_indices])]
    b_vector = torch.cat([neg_sorted, pos_sorted], dim=0).to(device)

    mask_small = w < v[0]
    mask_large = w > v[-1]
    mask_mid = (~mask_small) & (~mask_large)
    mask_edge = mask_small | mask_large

    idx_left = torch.zeros(M, dtype=torch.long, device=device)
    idx_right = torch.zeros(M, dtype=torch.long, device=device)
    theta = torch.zeros(M, dtype=torch.float32, device=device)

    if mask_edge.any():
        edge_idx = torch.nonzero(mask_edge, as_tuple=True)[0]
        w_edge = w[mask_edge]

        w_div_v0 = w_edge / v[0]
        w_div_v_last = w_edge / v[-1]

        edge_small = w_edge < v[0]
        edge_large = w_edge > v[-1]

        mask_cond_small = (w_div_v0 >= 0) & (w_div_v0 <= 1) & edge_small
        mask_else_small = edge_small & (~mask_cond_small)

        if mask_cond_small.any():
            inds = edge_idx[mask_cond_small]
            w_small = w_edge[mask_cond_small].unsqueeze(1)
            div_mat = w_small / v.unsqueeze(0)
            val_mat = div_mat * xi.unsqueeze(0)
            i_min = torch.argmin(val_mat, dim=1)
            vals_min = div_mat[torch.arange(i_min.shape[0], device=device), i_min]

            idx_left[inds] = i_min
            idx_right[inds] = i_min
            theta[inds] = vals_min

        if mask_else_small.any():
            inds = edge_idx[mask_else_small]
            idx_left[inds] = 0
            idx_right[inds] = 0
            theta[inds] = 1.0

        mask_cond_large = (w_div_v_last >= 0) & (w_div_v_last <= 1) & edge_large
        mask_else_large = edge_large & (~mask_cond_large)

        if mask_cond_large.any():
            inds = edge_idx[mask_cond_large]
            w_large = w_edge[mask_cond_large].unsqueeze(1)
            div_mat = w_large / v.unsqueeze(0)
            val_mat = div_mat * xi.unsqueeze(0)
            i_min = torch.argmin(val_mat, dim=1)
            vals_min = div_mat[torch.arange(i_min.shape[0], device=device), i_min]

            idx_left[inds] = i_min
            idx_right[inds] = i_min
            theta[inds] = vals_min

        if mask_else_large.any():
            inds = edge_idx[mask_else_large]
            idx_left[inds] = C - 1
            idx_right[inds] = C - 1
            theta[inds] = 1.0

    if mask_mid.any():
        mid_idx = torch.nonzero(mask_mid, as_tuple=True)[0]
        w_mid = w[mask_mid]

        ratio_b = w_mid[:, None] / v[b_vector]
        valid = (ratio_b >= 0) & (ratio_b <= 1) & (x_plus[b_vector] == 1).unsqueeze(0)

        valid_i0 = torch.where(
            valid,
            torch.arange(C, device=device)[None, :],
            torch.tensor(float("inf"), device=device)
        )

        i0_pos = valid_i0.argmin(dim=1)
        i0 = b_vector[i0_pos]
        v_i0 = v[i0]

        theta1 = w_mid / v_i0
        obj1 = theta1 * xi[i0]
        obj1 = torch.where(theta1 < 0, torch.tensor(float("inf"), device=device), obj1)

        one_indices = torch.nonzero(x_plus, as_tuple=True)[0].to(device=device, dtype=torch.long)
        i_right = torch.searchsorted(v[one_indices], w_mid, right=False)
        i_right = i_right.clamp(min=1, max=one_indices.shape[0] - 1)

        idx_r = one_indices[i_right]
        idx_l = one_indices[i_right - 1]

        v_left = v[idx_l]
        v_right = v[idx_r]

        theta2 = (w_mid - v_right) / (v_left - v_right + 1e-8)
        obj2 = theta2 * xi[idx_l] + (1.0 - theta2) * xi[idx_r]

        better_first = obj1 < obj2

        idx_left[mid_idx] = torch.where(better_first, i0, idx_l)
        idx_right[mid_idx] = torch.where(better_first, i0, idx_r)
        theta[mid_idx] = torch.where(better_first, theta1, theta2)

    denominator = v[idx_right] - v[idx_left]
    denominator_zero_mask = denominator == 0

    lambda_opt_nonzero = -(xi[idx_right] - xi[idx_left]) / denominator
    lambda_opt_zero_full = -xi / v
    lambda_opt_zero = lambda_opt_zero_full[idx_left]

    lambda_opt = torch.where(denominator_zero_mask, lambda_opt_zero, lambda_opt_nonzero)

    objective_values = delta + theta * xi[idx_left]

    mask_diff = idx_right != idx_left
    if mask_diff.any():
        objective_values[mask_diff] += (1.0 - theta[mask_diff]) * xi[idx_right[mask_diff]]

    x_placeholder = torch.stack(
        [idx_left.to(torch.int32), idx_right.to(torch.int32), theta],
        dim=1
    )

    return x_placeholder, lambda_opt, objective_values


def dense_from_leonardo_placeholder(x_placeholder, C, device):
    idx_left = x_placeholder[:, 0].to(dtype=torch.long, device=device)
    idx_right = x_placeholder[:, 1].to(dtype=torch.long, device=device)
    theta = x_placeholder[:, 2].to(dtype=torch.float32, device=device)

    M = x_placeholder.shape[0]
    rows = torch.arange(M, device=device)

    x_dense = torch.zeros(M, C, dtype=torch.float32, device=device)
    x_dense[rows, idx_left] += theta

    mask_diff = idx_right != idx_left
    x_dense[rows[mask_diff], idx_right[mask_diff]] += 1.0 - theta[mask_diff]

    return x_dense


def knapsack_solver_cvxpy(xi, v, w, C, delta):
    xi_np = xi.detach().cpu().numpy().astype(np.float64)
    v_np = v.detach().cpu().numpy().astype(np.float64)
    w_np = w.detach().cpu().numpy().astype(np.float64)
    delta_np = float(delta)

    M = w_np.shape[0]

    X_sol = np.zeros((M, C), dtype=np.float64)
    lambda_sol = np.zeros(M, dtype=np.float64)
    obj_sol = np.zeros(M, dtype=np.float64)

    installed = cp.installed_solvers()
    for candidate in ["CLARABEL", "ECOS", "OSQP", "SCS"]:
        if candidate in installed:
            solver = candidate
            break
    else:
        raise RuntimeError(f"Nessun solver CVXPY supportato trovato. Solver installati: {installed}")

    for i in range(M):
        x = cp.Variable(C)

        objective = cp.Minimize(xi_np @ x + delta_np * (1.0 - cp.sum(x)))

        constraints = [
            v_np @ x == w_np[i],
            cp.sum(x) <= 1.0,
            x >= 0.0,
            x <= 1.0,
        ]

        prob = cp.Problem(objective, constraints)

        if solver == "SCS":
            prob.solve(solver=solver, verbose=False, eps=1e-8, max_iters=200000)
        else:
            prob.solve(solver=solver, verbose=False)

        if prob.status not in ["optimal", "optimal_inaccurate"]:
            raise RuntimeError(
                f"CVXPY non ha trovato una soluzione per i={i}, "
                f"w={w_np[i]:.6f}, status={prob.status}"
            )

        X_sol[i] = np.where(np.abs(x.value) < 1e-8, 0.0, x.value)
        obj_sol[i] = prob.value

        lam = constraints[0].dual_value
        lambda_sol[i] = lam if lam is not None else np.nan

    return (
        torch.tensor(X_sol, dtype=torch.float32),
        torch.tensor(lambda_sol, dtype=torch.float32),
        torch.tensor(obj_sol, dtype=torch.float32),
        solver,
    )


def percentiles_like_trainer(x: torch.Tensor, probs: list[float]) -> torch.Tensor:
    """
    Stessa logica usata in trainer_on_gpus_pretrained.py.
    Usa kthvalue invece di torch.quantile.
    """
    x = x.flatten().float()
    n = x.numel()

    vals = []
    for p in probs:
        k = max(1, min(n, int(round(p * (n - 1))) + 1))
        vals.append(torch.kthvalue(x, k).values)

    return torch.stack(vals)


def make_v_from_w_percentiles_without_zero(w, C, device):
    """
    Costruisce C livelli di quantizzazione senza includere 0.
    Lo zero non è un bucket: è rappresentato dalla soluzione nulla x=0.
    """
    assert C >= 2, "Serve C >= 2."

    w = w.to(dtype=torch.float32, device=device)

    p_low, p_high = percentiles_like_trainer(w, [0.001, 0.999])

    eps = torch.tensor(1e-12, device=device)

    p_low = torch.minimum(p_low.to(device), -eps)
    p_high = torch.maximum(p_high.to(device), eps)

    if (p_high - p_low).abs() < 1e-12:
        center = 0.5 * (p_low + p_high)
        p_low = center - 1e-6
        p_high = center + 1e-6

    n_neg = C // 2
    n_pos = C - n_neg

    neg = torch.linspace(p_low.item(), -eps.item(), steps=n_neg, device=device)
    pos = torch.linspace(eps.item(), p_high.item(), steps=n_pos, device=device)

    v = torch.cat([neg, pos]).to(dtype=torch.float32)

    assert not torch.any(v == 0.0), f"Errore: v contiene 0: {v}"

    return v, p_low, p_high


device = torch.device("cpu")

C = 5
M = 100
delta = 100
r_w_generation = 0.20

xi = torch.randn(C, device=device)
xi = torch.sort(xi)[0]

w = -r_w_generation + 2.0 * r_w_generation * torch.rand(M, device=device)

# Inserisco alcuni zeri esatti per verificare che w=0 venga risolto con x=0
num_exact_zeros = min(10, M)
w[:num_exact_zeros] = 0.0

v, p_low, p_high = make_v_from_w_percentiles_without_zero(w, C, device)

w = torch.clamp(w, min=p_low.item(), max=p_high.item())
w = w[torch.randperm(M)]

print("=== INPUT ===")
print("xi    =", xi)
print("v     =", v)
print("p0.1  =", p_low)
print("p99.9 =", p_high)
print("w     =", w)
print("C     =", C)
print("delta =", delta)

x_sparse, lambda_leo, obj_leo_returned = knapsack_specialized_pruning_sparse_leonardo(
    xi.clone(), v.clone(), w.clone(), C, device, delta
)

x_leo_dense = dense_from_leonardo_placeholder(x_sparse, C, device)

tol = 1e-8

active_per_row = (torch.abs(x_leo_dense) > tol).sum(dim=1)

num_zero_active = (active_per_row == 0).sum().item()
num_one_active = (active_per_row == 1).sum().item()
num_two_active = (active_per_row == 2).sum().item()
num_more_than_two_active = (active_per_row > 2).sum().item()

print("\n=== SPARSITÀ x Leonardo ===")
print(f"righe con 0 componenti attive  = {num_zero_active}")
print(f"righe con 1 componente attiva  = {num_one_active}")
print(f"righe con 2 componenti attive  = {num_two_active}")
print(f"righe con >2 componenti attive = {num_more_than_two_active}")

obj_leo_recomputed = delta + x_leo_dense @ (xi - delta)

x_cvxpy, lambda_cvxpy, obj_cvxpy, solver_used = knapsack_solver_cvxpy(
    xi.clone(), v.clone(), w.clone(), C, delta
)

w_residual_leo = x_leo_dense @ v - w
sum_x_leo = x_leo_dense.sum(dim=1)

zero_w_mask = torch.abs(w) < 1e-12

if zero_w_mask.any():
    max_sum_x_at_zero_w = sum_x_leo[zero_w_mask].max().item()
    print(f"max sum(x) per w=0 = {max_sum_x_at_zero_w:.3e}")

max_x_err = torch.max(torch.abs(x_leo_dense - x_cvxpy)).item()
max_obj_err_returned = torch.max(torch.abs(obj_leo_returned.cpu() - obj_cvxpy)).item()
max_obj_err_recomputed = torch.max(torch.abs(obj_leo_recomputed.cpu() - obj_cvxpy)).item()
max_feas_err = torch.max(torch.abs(w_residual_leo)).item()
max_sum_violation = torch.clamp(sum_x_leo - 1.0, min=0.0).max().item()

print("\n=== CONFRONTO ===")
print(f"Solver CVXPY usato: {solver_used}")
print(f"max |x_leonardo_dense - x_cvxpy|             = {max_x_err:.3e}")
print(f"max |obj_returned_leonardo - obj_cvxpy|      = {max_obj_err_returned:.3e}")
print(f"max |obj_recomputed_from_sparse - obj_cvxpy| = {max_obj_err_recomputed:.3e}")
print(f"max |v @ x_leonardo - w|                     = {max_feas_err:.3e}")
print(f"max violation(sum(x) <= 1)                   = {max_sum_violation:.3e}")

if max_obj_err_recomputed < 1e-4 and max_feas_err < 1e-4 and max_sum_violation < 1e-4:
    print("\nOK: la x sparsa ricostruita è coerente con CVXPY.")
else:
    print("\nATTENZIONE: la x sparsa ricostruita NON coincide numericamente con CVXPY.")

if max_obj_err_returned >= 1e-4 and max_obj_err_recomputed < 1e-4:
    print(
        "Nota: objective_values ritornato dalla funzione non coincide, "
        "ma l'obiettivo ricalcolato dalla x sparsa sì. "
        "Questo segnala un possibile problema nella formula di objective_values "
        "quando idx_left == idx_right."
    )

df = pd.DataFrame({
    "w": w.detach().cpu().numpy(),
    "idx_left": x_sparse[:, 0].detach().cpu().numpy().astype(int),
    "idx_right": x_sparse[:, 1].detach().cpu().numpy().astype(int),
    "theta": x_sparse[:, 2].detach().cpu().numpy(),
    "obj_cvxpy": obj_cvxpy.detach().cpu().numpy(),
    "obj_returned_leonardo": obj_leo_returned.detach().cpu().numpy(),
    "obj_recomputed_from_sparse": obj_leo_recomputed.detach().cpu().numpy(),
    "abs_err_obj_returned": torch.abs(obj_leo_returned.cpu() - obj_cvxpy).numpy(),
    "abs_err_obj_recomputed": torch.abs(obj_leo_recomputed.cpu() - obj_cvxpy).numpy(),
})

display(df)

print("\n=== x Leonardo densa ===")
print(x_leo_dense)

print("\n=== x CVXPY ===")
print(x_cvxpy)

=== INPUT ===
xi    = tensor([-1.2051, -0.3355, -0.2205, -0.0301,  0.1456])
v     = tensor([-1.9862e-01, -1.0000e-12,  1.0000e-12,  9.9245e-02,  1.9849e-01])
p0.1  = tensor(-0.1986)
p99.9 = tensor(0.1985)
w     = tensor([ 0.1266, -0.1908,  0.0876,  0.0872,  0.0270,  0.0979, -0.1292, -0.1664,
         0.1257, -0.0360, -0.1741,  0.1380,  0.0835, -0.0302, -0.1509,  0.1857,
        -0.0680,  0.0000, -0.0533, -0.1920, -0.0590,  0.0783,  0.0080,  0.0000,
        -0.1188,  0.0425, -0.1305, -0.0082,  0.1289, -0.1878, -0.1697,  0.1146,
         0.1347,  0.1461,  0.0922,  0.0000,  0.1462,  0.0000, -0.0095,  0.0148,
         0.1687,  0.1536,  0.0916,  0.0000, -0.1547,  0.0732,  0.1237,  0.1985,
         0.0331,  0.0257,  0.1949, -0.0621, -0.1666, -0.0420, -0.1288,  0.0841,
         0.1918, -0.1986,  0.0000, -0.1815,  0.0095,  0.0000,  0.0836,  0.0187,
        -0.1702, -0.1734, -0.0003,  0.0078,  0.0594,  0.1083,  0.1610, -0.1136,
         0.1890, -0.1541,  0.0000, -0.0049, -0.1456,  0.0000,  0.03

,w,idx_left,idx_right,theta,obj_cvxpy,obj_returned_leonardo,obj_recomputed_from_sparse,abs_err_obj_returned,abs_err_obj_recomputed
0,0.126571,0,4,0.181105,-0.099043,-0.099045,-0.099045,1.907349e-06,1.907349e-06
1,-0.190825,0,4,0.980377,-1.178583,-1.178584,-1.178581,8.344650e-07,2.026558e-06
2,0.087637,0,4,0.279149,-0.231467,-0.231461,-0.231461,6.154180e-06,6.154180e-06
3,0.087152,0,4,0.280371,-0.233116,-0.233124,-0.233124,7.867813e-06,7.867813e-06
4,0.026955,0,4,0.431959,-0.437860,-0.437859,-0.437859,1.460314e-06,1.460314e-06
...,...,...,...,...,...,...,...,...,...
95,0.180611,0,4,0.045022,0.084758,0.084755,0.084755,3.322959e-06,3.322959e-06
96,0.017126,0,4,0.456712,-0.471292,-0.471291,-0.471291,1.043081e-06,1.043081e-06
97,0.174856,0,4,0.059513,0.065187,0.065186,0.065186,1.519918e-06,1.519918e-06
98,-0.025379,0,4,0.563747,-0.615860,-0.615860,-0.615860,5.960464e-08,5.960464e-08



=== x Leonardo densa ===
tensor([[0.1811, 0.0000, 0.0000, 0.0000, 0.8189],
        [0.9804, 0.0000, 0.0000, 0.0000, 0.0196],
        [0.2791, 0.0000, 0.0000, 0.0000, 0.7209],
        [0.2804, 0.0000, 0.0000, 0.0000, 0.7196],
        [0.4320, 0.0000, 0.0000, 0.0000, 0.5680],
        [0.2533, 0.0000, 0.0000, 0.0000, 0.7467],
        [0.8252, 0.0000, 0.0000, 0.0000, 0.1748],
        [0.9188, 0.0000, 0.0000, 0.0000, 0.0812],
        [0.1832, 0.0000, 0.0000, 0.0000, 0.8168],
        [0.5906, 0.0000, 0.0000, 0.0000, 0.4094],
        [0.9382, 0.0000, 0.0000, 0.0000, 0.0618],
        [0.1524, 0.0000, 0.0000, 0.0000, 0.8476],
        [0.2897, 0.0000, 0.0000, 0.0000, 0.7103],
        [0.5759, 0.0000, 0.0000, 0.0000, 0.4241],
        [0.8799, 0.0000, 0.0000, 0.0000, 0.1201],
        [0.0321, 0.0000, 0.0000, 0.0000, 0.9679],
        [0.6710, 0.0000, 0.0000, 0.0000, 0.3290],
        [0.4998, 0.0000, 0.0000, 0.0000, 0.5002],
        [0.6342, 0.0000, 0.0000, 0.0000, 0.3658],
        [0.9833, 0.0000,